# Pay Equity Analysis Using Skill Extraction

This notebook helps an HR analytics team analyze compensation data to ensure it aligns with the skills required for different roles, rather than just job titles.

- Author: Satya Phanindra Kumar Kalaga
- Date: September 2025


## 1. Setup

First, we'll install the necessary packages.

In [ ]:
!pip install laiser pandas matplotlib seaborn torch

## 2. Import Libraries

In [ ]:
import torch
import pandas as pd
from laiser.skill_extractor_refactored import SkillExtractorRefactored
import matplotlib.pyplot as plt
import seaborn as sns

## 3. Initialize the Skill Extractor

Initialize the `SkillExtractorRefactored`. Make sure to use your own Hugging Face model ID and token.

In [ ]:
# Replace with your Hugging Face model ID and token
model_id = "gemini"  # e.g., "mistralai/Mistral-7B-Instruct-v0.1"
api_key = "Your-key"

try:
    se = SkillExtractorRefactored(model_id=model_id, api_key=api_key, use_gpu=False)
    print("Skill Extractor initialized successfully!")
except Exception as e:
    print(f"Error initializing Skill Extractor: {e}")

## 4. Load Job and Compensation Data

We'll create a sample dataset that includes job descriptions and their corresponding salary information. In a real-world scenario, you would load this data from your HR systems.

In [ ]:
# Sample data representing job roles and compensation
data = {
    'job_id': [1, 2, 3, 4, 5, 6],
    'job_title': ['Junior Software Engineer', 'Senior Software Engineer', 'Data Analyst', 'Senior Data Analyst', 'HR Coordinator', 'HR Manager'],
    'description': [
        'A junior software engineer will assist in developing and maintaining software applications. Must know Python and basic SQL.',
        'A senior software engineer will lead development projects. Requires expertise in Python, SQL, cloud platforms like AWS, and containerization with Docker.',
        'The data analyst will be responsible for creating reports and dashboards. Proficiency in SQL and Tableau is required.',
        'The senior data analyst will lead analytics projects, requiring advanced SQL, Tableau, and some Python for scripting.',
        'The HR Coordinator will assist with daily HR tasks. Strong organizational skills and familiarity with HR software is a plus.',
        'The HR Manager will oversee the HR department. Requires experience in employee relations, compliance, and strategic planning.'
    ],
    'salary': [70000, 120000, 65000, 95000, 50000, 85000]
}
job_data = pd.DataFrame(data)
print("Job and compensation data loaded successfully:")
job_data

## 5. Extract Skills from Job Descriptions

Let's extract the skills from the job descriptions using the `laiser` package.

In [ ]:
try:
    extracted_skills = se.extract_and_align(
        job_data,
        id_column="job_id",
        text_columns=["description"],
        input_type='job_desc'
    )
    print("Skills extracted successfully!")
    print(extracted_skills.head())
except Exception as e:
    print(f"Error during skill extraction: {e}")

## 6. Analyze Skills and Compensation

Now, we'll analyze the relationship between the number of required skills and the salary for each role. This can help us spot potential inconsistencies in pay.

In [ ]:
if 'extracted_skills' in locals() and not extracted_skills.empty:
    # Count the number of skills for each job
    skill_counts = extracted_skills.groupby('Research ID').size().reset_index(name='skill_count')

    # Ensure Research ID is of integer type to match job_id
    skill_counts['Research ID'] = skill_counts['Research ID'].astype(int)

    # Merge skill counts with the original job data using left_on and right_on
    analysis_df = pd.merge(job_data, skill_counts, left_on='job_id', right_on='Research ID')

    # Visualize the relationship between skill count and salary
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=analysis_df, x='skill_count', y='salary', hue='job_title', s=200)
    plt.title('Salary vs. Number of Required Skills')
    plt.xlabel('Number of Skills')
    plt.ylabel('Salary')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Display the data table
    print("Data used for analysis:")
    print(analysis_df[['job_title', 'skill_count', 'salary']])
else:
    print("No skills were extracted to analyze.")

## 7. Identify Potential Pay Equity Issues

From the chart and table above, we can start to ask questions about pay equity. For example, if two roles have a similar number of required skills but a significant difference in salary, it might be worth investigating further.

This analysis provides a data-driven starting point for ensuring that compensation is fair and based on the actual skills required for a job.